# mem pipeline：各环节的想法、要求和缺口

**工作文档** · 建立于 2026-11-11 · 配套：[演示 notebook](../../report/2026-11-10_mem_pipeline_demo.ipynb)（看图和数字）、[讨论稿](../../docs/mem/mem分支_问题定义与讨论稿_2026-11-10.md)第 17 节（环节清单）

这份 notebook 只有文字，不需要运行。它把整条流程拆成 13 个环节，每个环节写清楚四件事：**想法**、**要求**、**现在怎么做**、**缺什么**，另附待定问题和建议先做的试验。

**怎么用**

- 每个环节后面有一个 **✏️ 你的更新** 格子，想法、修改意见、决定都可以写在那里；正文写得不对的地方也可以直接改。
- 改完保存，告诉我"读一下 pipeline"。我会对照上一次提交找出你改了什么，逐条回应，再把结论写回对应环节，并同步到 GitHub 和服务器。
- 我更新这个文件之前会先同步你的改动，不会覆盖你写的内容。

**状态标记**

| 标记 | 含义 |
| --- | --- |
| 真实 | 只用输入（视频、简单深度、设备位姿）算出 |
| ORACLE | 目前直接取自 SceneFun3D 的标注，以后要换成模型预测 |
| 部分 | 有一部分，核心内容还没有 |
| 缺 | 目前没有这个环节 |

"要求"都是我根据前两轮讨论起草的草案，不是已定的结论。数字全部来自场景 421069 的一个视频（42444490），只说明误差从哪里来，不是结果。

## 已定的约定（前两轮讨论）

这些是各环节共同的前提，改动任何一条都会影响多个环节。

- **记忆从视频构建。** 输入只有 RGB、简单深度和设备位姿；激光扫描、由扫描渲染的深度、扫描配准的位姿只用于制作训练标签和评测。
- **单目深度为主**，用设备 LiDAR 校正尺度；纯 LiDAR 作为对照。允许使用设备位姿（ARKit、Aria）。
- **动作词表**沿用 SceneFun3D 的 9 类：rotate、key_press、tip_push、hook_pull、pinch_pull、hook_turn、foot_push、plug_in、unplug。
- **人形只用于展示**，但"怎么接近、位置准、动作对"是展示的核心：affordance 是为之后的动作做准备。
- **不自采数据**；可以使用 SMPL-X 和仅限研究用途的公开数据。
- **阶段依次进行：** S1（SceneFun3D 静态视频）→ S2（FunGraph3D 交互视频，检验交互能否帮助构建记忆）→ S3（HD-EPIC，检验事件结构能否帮助检索）。
- **流程确认之前不做正式训练和完整评测**；GPU 作业只通过服务器队列运行。

## 总览

```text
输入        1 关键帧选取 ── 2 深度 ── 3 相机位姿
              ↓
单帧理解    4 部件定位与动作类型（affordance map） ── 5 部件运动
              ↓
记忆构建    6 跨帧关联与合并 ── 7 接触点 ── 8 法向、地面与站立点 ── 9 条目内容
              ↓
查询        10 查询与检索
              ↓
人形展示    11 末端姿态 ── 12 走近过程与画面

评测        13 评测（贯穿所有环节）
```

| # | 环节 | 状态 | 最主要的缺口 |
| --- | --- | --- | --- |
| 1 | 关键帧选取 | 真实 | 38 个部件有 9 个没进记忆，原因没查清 |
| 2 | 深度 | 真实 | 部件处误差 11.2 cm，是全图的两倍多 |
| 3 | 相机位姿 | 真实 | 个别帧误差大，没有任何校正 |
| 4 | 部件定位与动作类型 | ORACLE | 没有模型，也没有批量制作训练标签的流程 |
| 5 | 部件运动 | ORACLE | 没有预测方法 |
| 6 | 跨帧关联与合并 | 缺 | 整个环节 |
| 7 | 接触点 | 真实 | 只用一帧；单目深度下只有 14% 落在部件上 |
| 8 | 法向、地面与站立点 | 真实 | 没看到的空间被当成空闲；不检查手是否够得着 |
| 9 | 条目内容 | 部分 | "动作结果"还没有自己的表示 |
| 10 | 查询与检索 | ORACLE | 没有真正的检索，也没有对照组 |
| 11 | 末端姿态 | 真实 | 9 类动作只有伸手和伸脚两种姿态 |
| 12 | 走近过程与画面 | 真实 | 不避障，脚会滑 |
| 13 | 评测 | 真实，单场景 | 只有一个场景；缺检索、运动和官方指标 |

## 1 · 关键帧选取

**状态：** 真实

**想法：** 记忆以关键帧为单位，不做全局重建。从视频里挑少量帧，后面所有环节只在这些帧上计算。

**输入 → 输出：** 整段视频（RGB，10 FPS，1025 帧）→ 一组关键帧。

**要求（草案）：**
- 不能借助标注来选帧，否则等于提前知道部件在哪里。
- 每个功能部件至少在一帧里足够大、足够清楚。
- 帧数尽量少，保证后面的计算量小。
- 同一方法能用于 S2 的交互视频和 S3 的长视频。

**现在：** 按时间均匀取 40 帧，工作分辨率 720×960（原始分辨率的一半）。为了能打分，只在有真值深度的帧里取。代码：`report/mem_demo_helpers.py` 的 `pick_keyframes`。

**缺什么：**
- 38 个部件里有 9 个在这 40 帧中可见像素不足 10 个，没有进入记忆。还没查清是视频本来就没拍清楚，还是关键帧取少了。
- 没有按相机移动量或画面内容选帧，也没有剔除模糊帧。
- 没有测过帧数和部件覆盖率的关系。

**待定：**
- 取多少帧。
- 选帧方式：按时间均匀、按相机移动量，还是按"出现了新内容"。
- 小部件处是否需要回到原始分辨率。

**建议先试：** 画出帧数（20、40、80、160、全部）对部件覆盖率的曲线，先回答那 9 个部件是不是视频里本来就看不清。只用 CPU。

### ✏️ 1 · 你的更新

（写在这一行下面）

## 2 · 深度

**状态：** 真实

**想法：** 用"简单深度"代替重建：每个关键帧一张深度图，把像素抬到三维。主用单目深度，因为任何视频都能用；设备 LiDAR 只用来校正尺度，并作为对照。

**输入 → 输出：** 关键帧 RGB，加上低分辨率 LiDAR 深度（256×192）→ 每帧一张以米为单位的深度图。

**要求（草案）：**
- 尺度正确，单位是米。
- 在小部件及其周围足够准。部件只有几厘米宽，深度差几厘米，接触点就落到部件外面。
- 不依赖激光扫描。
- 没有 LiDAR 的视频也要有办法定尺度。S3 的 HD-EPIC 由 Aria 眼镜拍摄，没有深度传感器。

**现在：** Depth Anything V2（室内米制版，Large）。它的原始输出偏远约 33%，所以每帧乘一个尺度（LiDAR 与预测之比的中位数）。校正后全图中位误差 4.7 cm，部件处 11.2 cm。纯 LiDAR 是 2.4 cm 和 5.2 cm，但只覆盖 88% 的像素。代码：`affordance3d/mem/depth_models.py`，`pipeline.py` 的 `lidar_scaled`。

**缺什么：**
- 部件处误差是全图的两倍多，原因没有拆开：是细小结构被抹平，还是一个全局尺度不够。
- 只校正了一个尺度，没有校正偏移，也没有分区域校正。
- 没有比较其他深度模型。Depth Pro 在 Mac 上输出不可信，需要在服务器上重测；MoGe-2、UniDepth V2 没试。
- 没有 LiDAR 时怎么定尺度，没试。

**待定：** 小部件处用哪种办法。
- A：部件附近优先用 LiDAR，空洞处用单目深度补。
- B：用 LiDAR 做"尺度 + 偏移"或分区域校正。
- C：不追求部件本身的深度，由周围表面推算（见环节 7）。
- D：以后把接触点深度作为模型的预测目标。

**建议先试：** 在部件周围比较 A、B、C 的误差，和环节 7 一起做。只用 CPU。

### ✏️ 2 · 你的更新

（写在这一行下面）

## 3 · 相机位姿

**状态：** 真实

**想法：** 用设备自带的位姿（ARKit）把各帧放进同一个坐标系，不做 SfM 或 SLAM。

**输入 → 输出：** ARKit 轨迹（60 FPS）→ 每个关键帧的相机位姿和重力方向。

**要求（草案）：**
- 同一个部件从不同帧抬到三维后要落在一起（厘米级），否则环节 6 无法判断是不是同一个部件。
- 高度方向不能漂，否则地面和站立点会错。
- 只用设备数据。

**现在：** 直接使用。与真值相比，相机中心误差中位 7.2 cm，个别帧 89 cm；朝向误差中位 1.2°。高度漂移使人形双脚离真实地面 2–6 cm。为了打分，输入相机用真值配准变换到扫描坐标系，这是整体的刚体变换，不改变帧与帧之间的相对关系。代码：`affordance3d/mem/inputs.py`。

**缺什么：**
- 没有分析误差在时间上的分布：哪些帧差，是否集中在某一段。
- 没有任何校正，也没有剔除明显有问题的帧。
- 没有直接量过环节 6 真正需要的量：同一个部件在不同帧里抬到三维后相差多少。

**待定：**
- 接受现状，还是在关键帧之间做轻量校正（例如用深度对齐相邻帧）。
- 是否剔除位姿明显有问题的帧。

**建议先试：** 借用标注的部件编号，统计同一部件在各关键帧里三维位置的散布。如果散布明显小于相邻部件的间距，这个环节先不动。

### ✏️ 3 · 你的更新

（写在这一行下面）

## 4 · 部件定位与动作类型（affordance map）

**状态：** ORACLE

**想法：** 这就是 affordance map 本身：在每个关键帧上标出哪里可以操作、属于哪一类动作。它和深度一起，是记忆的全部原料。

**输入 → 输出：** 关键帧 RGB（可选深度）→ 每帧里各部件的位置，以及每个部件的动作类型（9 类）。

**要求（草案）：**
- 能找到小部件：把手、旋钮、按键、插座。
- 位置准到能确定接触点。
- 动作类型正确。
- 只看这一帧或邻近几帧，不看扫描。
- 在没见过的场景上有效。SceneFun3D 有官方的训练和验证划分。
- 能与公开方法对照。

**现在：** ORACLE。把扫描上标注的部件点用真值相机投影到帧上（深度容差 5 cm），动作类型取标注。得到的是稀疏的点，不是完整的掩码。代码：`affordance3d/mem/labels.py`。

**缺什么：**
- 模型本身。
- 批量制作训练标签的流程：现在只对一个场景做了投影。
- 稀疏点可能需要补成完整掩码。
- 这一环节自己的评测指标。

**待定：**
- 输出形式：每类动作一张热力图、每个部件一个掩码，还是每个部件一个点。
- 模型路线：在预训练视觉特征上训练小预测头，还是先看现成的分割或视觉语言模型不训练能做到什么程度。
- 训练规模和算力。训练需要 GPU，要走队列。

**建议先试：** 不训练，先看现成模型在这 40 帧上能找到多少部件，作为基线；同时把标签制作做成可以批量运行的。

### ✏️ 4 · 你的更新

（写在这一行下面）

## 5 · 部件运动

**状态：** ORACLE

**想法：** 记住部件怎么动：转动还是平移，绕哪根轴，朝哪个方向。它是"动作结果"的几何部分，也决定人形手的运动方向。

**输入 → 输出：** 部件在帧上的位置和深度 → 运动类型、轴的方向、轴的原点。

**要求（草案）：**
- 类型正确（转动或平移）。
- 方向准到画面上看起来对：抽屉向外，门绕铰链。
- 从输入得到。

**现在：** ORACLE。取自 SceneFun3D 的运动标注，38 个部件里 37 个有。演示里显示为黄色箭头。

**缺什么：**
- 预测方法。
- 误差度量还没用起来（轴夹角的函数已有：`actionable.py` 的 `axis_angle_deg`）。
- 数据集里没有运动范围（能开多大）。
- S1 是静态视频，看不到部件真的在动，只能从外观推断；S2 的交互视频里才有真实运动。

**待定：**
- S1 先保留标注，到 S2 再从交互视频里估计，还是 S1 就从外观预测。前一种正好是"交互帮助构建记忆"的检验点。
- 表示方式：直接回归方向，还是利用表面法向和重力来约束。

**建议先试：** 检查标注的轴是否几乎总是平行于重力、表面法向，或两者的叉积。如果是，运动可以由"类型 + 表面几何"得到，不必回归任意方向。只用 CPU。

### ✏️ 5 · 你的更新

（写在这一行下面）

## 6 · 跨帧关联与合并

**状态：** 缺

**想法：** 同一个部件会出现在多帧里。记忆里应当只有一个条目，并把多帧的证据合起来，这样更准，也更可信。

**输入 → 输出：** 各帧里的部件（位置、动作类型、三维点）→ 部件条目，每个条目带一组证据帧。

**要求（草案）：**
- 同一个部件不重复建条目；相邻的不同部件（一排抽屉把手、一排开关）不被合并。
- 多帧合并后比单帧准。
- 给出置信度：见过几次，各帧是否一致。
- 能随时间增量更新，S2 和 S3 需要。

**现在：** 没有这个环节。标注自带部件编号，所以不需要判断。每个条目只用像素最多的那一帧，其他帧只被记下来，没有参与计算。

**缺什么：** 全部：关联规则、合并方式、置信度。

**待定：**
- 关联的依据：三维距离、外观特征、动作类型是否一致。
- 距离阈值怎么定。它受位姿误差（环节 3）限制。
- 合并方式：取中位数，还是按像素数或距离加权。

**建议先试：** 把标注的部件编号藏起来，只按三维位置聚类，看能否恢复出原来的部件（有真值，可以直接算对错）；同时看多帧取中位数能把接触点误差降多少。只用 CPU。

### ✏️ 6 · 你的更新

（写在这一行下面）

## 7 · 接触点

**状态：** 真实

**想法：** 手或脚应当碰到的那个三维点，是"位置准确"的核心量。

**输入 → 输出：** 部件在帧上的像素、深度、位姿 → 一个三维点。

**要求（草案）：**
- 落在部件上。目前的标准是离真值部件不超过 3 cm。
- 对深度噪声稳健。
- 在人够得着的那一面，不在部件背面或内部。

**现在：** 取部件像素的中心像素，深度取部件像素深度的中位数，只用像素最多的一帧。代码：`pipeline.py` 的 `_contact_and_normal`。

| 用什么深度和位姿 | 进入记忆的部件 | 中位误差 | 落在部件上 |
| --- | --- | --- | --- |
| 单目深度 × LiDAR 尺度，ARKit 位姿 | 29 | 7.7 cm | 14% |
| LiDAR，ARKit 位姿 | 24 | 2.8 cm | 54% |
| 真值深度，真值位姿 | – | 0.4 cm | 97% |

最后一行说明计算链路本身是对的，误差来自深度和位姿。个别部件很差：烤面包机的旋钮 42.8 cm、按键 64.7 cm，都是只在一帧里出现的很小的部件。

**缺什么：**
- 多帧融合，依赖环节 6。
- 没有利用周围的表面。
- 没有处理明显错误的深度。
- 接触点现在是部件中心，没有考虑该抓哪里，例如门把手应当握在远离转轴的一端。

**待定：**
- 提高精度的办法：A 多帧取中位数；B 沿视线与周围支撑面求交，再加一个小偏移；C 优先用 LiDAR；D 以后由模型预测。
- 接触点的定义：部件中心，还是随动作类型而定。

**建议先试：** A、B、C 单独和组合的误差表。只用 CPU。

### ✏️ 7 · 你的更新

（写在这一行下面）

## 8 · 法向、地面与站立点

**状态：** 真实

**想法：** 人形从哪里、朝哪个方向接近部件。法向给出部件朝外的方向，地面给出脚的高度，站立点是人站的位置。

**输入 → 输出：** 接触点、周围的深度、所有关键帧观察到的点 → 支撑面法向、地面高度、站立位置和朝向、是否空闲。

**要求（草案）：**
- 站立点在地面上，没有被占据，而且确实被看到过是空的。
- 从站立点手够得到接触点，必要时前倾或蹲下。
- 朝向面对部件。
- 部件在台面上方或朝上时，也有合理的站位。

**现在：**
- 法向：接触点周围 48×48 像素内（去掉部件像素）拟合平面。
- 地面：最低 20% 的点里，高度出现最多的那一档（每档 2 cm）。
- 站立点：沿水平法向后退 0.5 m（用脚的动作 0.45 m）；被挡就再退 0.15、0.3、0.45 m，还不行就转 ±30°、±60°。法向接近竖直时，改为从拍摄方向接近。
- 是否空闲：身体当作半径 0.2 m 的圆柱，与观察到的点（5 cm 体素）比较。
- 结果：站立点在真值扫描里空闲的比例是 93%（单目深度）和 75%（LiDAR）。代码：`affordance3d/mem/actionable.py`。

**缺什么：**
- 没看到的空间被当成空闲，与"未知不等于空闲"的原则不符。
- 不检查够不够得着：退得越远越够不着。
- 拟合窗口按像素而不是按米，远处部件的窗口实际很大。
- 法向没有稳健拟合，窗口跨两个面时会歪；也没有量过法向误差（真值可以从扫描算）。
- 地面只有一个高度，有台阶的场景不适用。

**待定：**
- 体素是否分成空闲、占据、未知三种。
- 够得着的检查用什么形式。
- 多个候选站立点如何打分（距离、朝向、空闲程度）。

**建议先试：** 加上三态体素和够得着的约束，重新统计"空闲且够得着"的比例。只用 CPU。

### ✏️ 8 · 你的更新

（写在这一行下面）

## 9 · 条目内容

**状态：** 部分

**想法：** 记忆里每个部件存什么。这是最初想法落地的地方：不只存语义，还存这个部件可能的动作和动作的结果。

**输入 → 输出：** 前面各环节的输出 → 一条记录。

**要求（草案）：**
- 能回答五个问题：在哪里；能做什么；做了会怎样；怎么做（接触点、方向、站位）；凭什么（证据帧）。
- 支持环节 10 的各类查询。
- 每个字段都能从输入得到或由模型预测，并且能评测。
- 为 S2、S3 留位置：事件（谁在何时做了什么）和状态随时间的变化。

**现在的字段：** 动作类型、任务句子（取自标注）、最佳关键帧和像素、所有见过它的帧、接触点、法向、站立点（位置、朝向、是否空闲）、运动（取自标注）。记忆整体还存关键帧列表、地面高度和观察到的点。代码：`pipeline.py` 的 `MemoryEntry` 和 `Memory`。

**缺什么：**
- **动作结果。** 现在由 SceneFun3D 的任务句子充当（如"打开落地灯"），它是标注，不是记忆自己得到的。没有状态变量（开或关、锁或未锁），也没有作用对象（开关 → 灯）。
- 检索用的特征（视觉、文本）。
- 所属物体和它的语义：这个把手属于哪个柜子。
- 置信度。
- 时间：什么时候看到的。

**待定：**
- 动作结果怎么表示：A 一句由模型生成的话；B 结构化（状态变量、作用对象、前后状态）；C 两者都存。
- S1 里的结果只能是"预测的可能结果"，S2、S3 才有"观察到的结果"。两者是否分开存。
- 是否需要所属物体。需要的话，要增加物体级的分割。

**这个环节不需要试验，需要先定方向。** 定了之后才知道环节 4、5 要预测什么，环节 10 能查什么。

### ✏️ 9 · 你的更新

（写在这一行下面）

## 10 · 查询与检索

**状态：** ORACLE

**想法：** 用一句话从记忆里找到要操作的部件、怎么操作、以及证据帧；并检验用动作信息检索是否比只用语义的静态检索好。

**输入 → 输出：** 文字查询 → 条目，附证据关键帧和人形画面。

**要求（草案）：**
- 支持按结果查（"打开落地灯"）、按动作查（"哪些东西可以拉开"）、按位置或物体查。
- 返回的结果带证据帧。
- 有公平的对照：只用语义特征的静态检索（讨论稿第 4 节的对照组）。
- 有真值可评：SceneFun3D 标注了每条任务句子对应哪个部件。

**现在：** ORACLE。查询必须与任务句子完全相同，直接按标注的对应关系返回。这个场景有 24 条任务句子。代码：`pipeline.py` 的 `retrieve`。

**缺什么：**
- 真正的检索：特征、相似度、排序。
- 评测：命中率。
- 对照组的实现。
- 自由文本查询。

**待定：**
- 先支持哪类查询。
- 条目用什么做检索键：部件图像块的图文特征；条目的文字描述；是否加动作类型过滤。
- 查询时是否允许视觉语言模型重新看证据帧。

**建议先试：** 用现成的图文特征对 24 条任务句子做检索，比较"只用外观"和"外观 + 动作类型"的命中率。这是"动作信息是否帮助检索"在 S1 上的第一个数。只用 CPU。

### ✏️ 10 · 你的更新

（写在这一行下面）

## 11 · 末端姿态

**状态：** 真实

**想法：** 人形最后的姿势：站在站立点，面对部件，手或脚到接触点。画面要让人一眼看出位置准、动作对。

**输入 → 输出：** 条目（接触点、法向、站立点、动作类型）和地面高度 → SMPL-X 姿态。

**要求（草案）：**
- 手或脚落在部件上。
- 姿态自然：不穿过家具，关节不反折。
- 不同的动作类型看得出区别。
- 脚在地面上。

**现在：** 手掌瞄准接触点前方 3 cm。接触点低于 0.55 m 时蹲下。部件偏左用左手，否则用右手；foot_push 用右脚。姿态由 300 步迭代求出，只调脊柱和一条手臂（或一条腿）。没有手指动作。演示的 4 个查询里，手或脚到真值部件 1.6–8.6 cm，脚离真实地面 2.3–5.5 cm，身体与扫描的冲突不超过 0.1%。代码：`affordance3d/mem/humanoid.py`。

**缺什么：**
- 9 类动作只有伸手和伸脚两种姿态，捏、勾、按、转没有区别。
- 手的朝向没有约束，手掌不一定对着表面。
- 不演示动作本身：沿运动轴拉、转。
- 隔着台面够不到时，不会前倾或侧身。
- 穿模只是事后统计，没有作为约束。
- 只有一种体型。

**待定：**
- 每类动作一个手型预设是否足够。
- 是否加动作演示：手沿运动轴移动一小段，部件本身不动。
- 现在做，还是等生成式动作（环节 12）一起做。

**建议先试：** 给 9 类动作各做一个手型，在同一个部件上并排画出来比较。只用 CPU。

### ✏️ 11 · 你的更新

（写在这一行下面）

## 12 · 走近过程与画面

**状态：** 真实

**想法：** 人形怎么走到部件前，以及最后给人看的画面。

**输入 → 输出：** 站立点、末端姿态、视频帧 → 一段动画和静态图。

**要求（草案）：**
- 走的路径不穿过家具。
- 步态自然，脚不滑。
- 画面里同时看得到人和部件，遮挡关系正确。
- 能批量生成，每个查询一段。

**现在：** 起点在站立点朝相机方向最多 1.5 m 处。直线走 18 帧，再用 12 帧伸手。步态是简单的周期摆动。展示帧从整段视频里挑选，标准是部件可见、人在画面内、距离合适。人体用输入深度做遮挡。另有一个在扫描里的检查视角。代码：`humanoid.py` 的 `approach_frames`，`report/mem_demo_helpers.py`。

**缺什么：**
- 路径规划：现在不避障。
- 真实的步态：现在脚会滑。
- 从走到伸手的过渡是线性插值。
- 背景是静止的一帧，相机不动。
- 生成式动作模型（LINGO、DartControl）还没有安装。

**待定：**
- 是否换成生成式动作。需要 GPU，要走队列。
- 画面形式：贴在真实视频帧上、在三维里自由视角，还是两者并排。
- 需要细致到什么程度。

**建议先试：** 先定画面形式，再决定是否上生成式动作。

### ✏️ 12 · 你的更新

（写在这一行下面）

## 13 · 评测

**状态：** 真实，单场景

**想法：** 用激光扫描和标注给每个环节打分，并回答核心问题：加入动作信息的记忆是否更好。

**要求（草案）：**
- 每个环节有自己的指标，能看出误差从哪里来。
- 多个场景，有训练和测试的划分。
- 与公开方法可比。
- 有对照组（讨论稿第 4 节）。

**现在：** 一个场景，一个视频。指标有：接触点误差、落在部件上的比例（不超过 3 cm）、站立点在扫描里是否空闲、手或脚到部件的距离、脚离地高度、身体与扫描的冲突。代码：`report/mem_demo_helpers.py` 的 `scoreboard` 和 `shots_table`。

**缺什么：**
- 多场景。
- 换成预测后需要的指标：部件发现率、动作类型准确率、运动轴误差、检索命中率。
- SceneFun3D 官方指标。它们定义在激光扫描的点上，我们需要把预测投回扫描才能比；而且公开方法以扫描为输入，设定不同。
- 对照组的实现。

**待定：**
- 开发阶段用多少个场景。
- 哪些指标作为主指标。

**这个环节随每个环节一起补，不单独做。**

### ✏️ 13 · 你的更新

（写在这一行下面）

## 依赖和建议顺序

- **环节 9 最先定。** 它决定环节 4、5 要预测什么，也决定环节 10 能查什么。
- **环节 2、6、7 一起看。** 三者共同决定接触点精度，这是目前最大的误差。
- **现在就能试（只用 CPU）：** 1、2、3、5 的轴检查、6、7、8、10、11。
- **需要 GPU，要走队列：** 4 的训练，12 的生成式动作。

建议顺序：9 → 2 和 7 → 6 → 8 → 10 → 4 和 5 → 11 和 12；环节 13 随每个环节一起补。顺序由你定。

### ✏️ 总体更新

不属于某一个环节的想法、顺序上的调整、新增或合并环节，都写在这里。

（写在这一行下面）